Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 14 — audyt błędów i interpretacja decyzji (EuroSAT + ResNet18)

Materiał towarzyszący [`README.pl.md`](README.pl.md). Inny problem niż wykład
(tam: Fashion-MNIST od zera). Tu: EuroSAT + ResNet18 zamrożony backbone.

In [ ]:
import collections  # noqa: F401 - uzywane w TODO ponizej
import time

import matplotlib.pyplot as plt  # noqa: F401 - uzywane w TODO ponizej
import torch
import torch.nn as nn
import torch.nn.functional as F  # noqa: F401 - uzywane w TODO ponizej
import torchvision.datasets as tvd
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")

weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
BASELINE_GRAY = (0.5 - MEAN) / STD  # mid-gray in normalized space, for occlusion later

full_ds = tvd.EuroSAT(root="data_cache", download=True, transform=preprocess)
CLASS_NAMES = full_ds.classes
print(f"total images: {len(full_ds)}, classes: {CLASS_NAMES}")


def load_subset(indices):
    X = torch.stack([full_ds[i][0] for i in indices])
    y = torch.tensor([full_ds[i][1] for i in indices])
    return X, y


backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for p in backbone.parameters():
    p.requires_grad = False
N_CLASSES = 10
backbone.fc = nn.Linear(backbone.fc.in_features, N_CLASSES)
backbone.eval()
print(f"trainable params: {sum(p.numel() for p in backbone.parameters() if p.requires_grad)}")


def extract_features(model, X, batch_size=64):
    feats = []
    orig_fc = model.fc
    model.fc = nn.Identity()
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            feats.append(model(X[i:i + batch_size]))
    model.fc = orig_fc
    return torch.cat(feats, dim=0)

## 1. Dane i podział train/validation/test — Twoje zadanie

In [ ]:
# TODO:
# 1. torch.manual_seed(0); perm = torch.randperm(len(full_ds))
# 2. train_idx, val_idx, test_idx = perm[:1800], perm[1800:2200], perm[2200:2600]
#    (lub inny rozlaczny podzial, uzasadnij rozmiary)
# 3. ZWERYFIKUJ PROGRAMOWO brak nakladania (np. przez set intersection)
# 4. wypisz rozklad klas w kazdym podzbiorze (collections.Counter)
# 5. komentarz: co by sie stalo, gdybys uzyl tego samego podzialu do wyboru
#    hiperparametrow I do finalnej oceny?
raise NotImplementedError("TODO: podzial danych bez przecieku")

## 2. Trening głowicy (gotowe + krótkie zadanie)

In [ ]:
def train_head(feat_train, y_train, n_classes=10, epochs=30, batch_size=64, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    head = nn.Linear(feat_train.shape[1], n_classes)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = feat_train.shape[0]
    t0 = time.time()
    for epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            out = head(feat_train[idx])
            loss = loss_fn(out, y_train[idx])
            loss.backward()
            opt.step()
    train_time = time.time() - t0
    return head, train_time

In [ ]:
# TODO:
# X_train, y_train = load_subset(train_idx.tolist())  # noqa: F821
# X_val, y_val = load_subset(val_idx.tolist())  # noqa: F821
# X_test, y_test = load_subset(test_idx.tolist())  # noqa: F821
# feat_train = extract_features(backbone, X_train)
# feat_val = extract_features(backbone, X_val)
# feat_test = extract_features(backbone, X_test)
# head, train_time = train_head(feat_train, y_train)  # TRAIN ONLY on train_idx!
# backbone.fc = head
# print(f"train_time: {train_time:.2f}s")
raise NotImplementedError("TODO: wywolaj trening glowicy na train_idx")

## 3. Ewaluacja na niezależnym zbiorze — Twoje zadanie

In [ ]:
# TODO:
# with torch.no_grad():
#     test_logits = backbone.fc(feat_test)  # head already trained, reuse precomputed features
#     test_pred = test_logits.argmax(1)
#     test_probs = F.softmax(test_logits, dim=1)
# test_acc = (test_pred == y_test).float().mean().item()
#
# C = 10
# confusion = torch.zeros(C, C, dtype=torch.long)
# for t, p in zip(y_test.tolist(), test_pred.tolist()): confusion[t, p] += 1
#
# per-class precision/recall/F1, macro-F1 -- reuse the pattern from the lecture
# plt.imshow(confusion.numpy(), cmap="Blues") + labels + colorbar
raise NotImplementedError("TODO: ewaluacja i confusion matrix na test_idx")

## 4. Najczęstsze i najkosztowniejsze błędy — Twoje zadanie

In [ ]:
# TODO: z confusion matrix (bez przekatnej), znajdz DWIE najczesciej mylone pary
# klas. Komentarz: czy jedna z tych par ma inny "koszt" pomylenia niz inna?
raise NotImplementedError("TODO: najczestsze bledy")

## 5. Confidence poprawnych i błędnych predykcji — Twoje zadanie

In [ ]:
# TODO:
# confidences = test_probs.max(dim=1).values
# correct_mask = test_pred == y_test
# histogram confidences[correct_mask] vs confidences[~correct_mask]
# znajdz: idx najbardziej pewnej BLEDNEJ predykcji (confidences[~correct_mask].argmax(),
# pamietaj przeliczyc indeks z powrotem na pelny test_idx)
raise NotImplementedError("TODO: analiza confidence")

## 6. Wybór próbek do szczegółowej analizy — Twoje zadanie

In [ ]:
# TODO: zdefiniuj JAWNA procedure (zapisz w komentarzu DOKLADNIE, jak kazdy
# przyklad zostal wybrany):
# a) poprawna predykcja o WYSOKIEJ confidence
# b) pomylka o NISKIEJ confidence
# c) pomylka o WYSOKIEJ confidence (mylaca)
raise NotImplementedError("TODO: wybor probek do analizy")

## 7. Grad-CAM — Twoje zadanie

Backbone jest zamrożony -- `model.eval()` nie wystarcza, potrzebujesz
`x.requires_grad_(True)` na wejściu (patrz wykład).

In [ ]:
# TODO:
# class GradCAM:
#     def __init__(self, model, target_layer):
#         ...  # forward_hook + register_full_backward_hook
#     def __call__(self, x, target_class):
#         ...  # alpha = grad.mean(dim=(2,3)); weighted sum; relu; interpolate; normalize
#              # (handle zero-spread map: check (max-min) > eps before dividing)
#     def remove_hooks(self):
#         ...
raise NotImplementedError("TODO: implementacja Grad-CAM")

## 8. Porównanie heatmap i occlusion sensitivity — Twoje zadanie

In [ ]:
# TODO:
# 1. dla kazdego z 3 przykladow (sekcja 6): policz Grad-CAM na backbone.layer4
#    dla klasy PRZEWIDZIANEJ, na x.clone().requires_grad_(True)
# 2. narysuj obraz + heatmapa + overlay dla kazdego
# 3. dla przykladu (a) [poprawna, wysoka confidence]: policz occlusion sensitivity
#    (patch=32, stride=32, baseline=BASELINE_GRAY -- NIE zero)
# 4. komentarz: regiony Grad-CAM i occlusion sie zgadzaja czy roznia?
raise NotImplementedError("TODO: Grad-CAM + occlusion na wybranych probkach")

## 9. Analiza podgrup LUB odporność — Twoje zadanie (wybierz JEDNO)

Zobacz `README.pl.md` sekcja 9 dla pełnych instrukcji obu opcji.

In [ ]:
# TODO: wybierz OPCJE A (jasnosc) albo OPCJE B (flip horyzontalny), zaimplementuj,
# i odpowiedz na pytanie krytyczne z README (konfundowanie klasa / generalizacja
# na rzeczywisty distribution shift)
raise NotImplementedError("TODO: analiza podgrup lub odpornosci")

## 10. Zadanie diagnostyczne

Poniższy kod (gotowy, uruchom go) próbuje policzyć Grad-CAM, ale ma dwa
powiązane problemy.

In [ ]:
class GradCAMColleague:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, target_class):
        # BUG 1: wrapped in no_grad "for speed" by a well-meaning colleague
        with torch.no_grad():
            self.model.zero_grad()
            logits = self.model(x)
            score = logits[:, target_class].sum()
            score.backward()
        return self.gradients

    def remove_hooks(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()


gradcam_colleague = GradCAMColleague(backbone, backbone.layer4)
try:
    result = gradcam_colleague(X_test[0:1], target_class=0)  # noqa: F821
    print("no error (unexpected)")
except RuntimeError as e:
    print(f"RuntimeError: {e}")

In [ ]:
# TODO:
# 1. komentarz: dokladnie jaki wyjatek, i dlaczego (odwolaj sie do wykladu)?
# 2. napraw BUG 1: usun torch.no_grad() z __call__ -- uruchom ponownie, wciaz blad?
# 3. zdiagnozuj BUG 2 (zamrozony backbone + brak requires_grad na wejsciu)
# 4. napraw OBA, zweryfikuj ze gradient jest skonczony i niezerowy dla
#    przynajmniej jednego z Twoich przykladow z sekcji 6
raise NotImplementedError("TODO: diagnoza i poprawka obu problemow")

## 11. Raport końcowy — Twoje zadanie

**1. Jak dobrze model klasyfikuje poszczególne klasy? (konkretne liczby)**

_(Twoja odpowiedź tutaj.)_

**2. Które pomyłki są najczęstsze? (konkretne pary, z confusion matrix)**

_(Twoja odpowiedź tutaj.)_

**3. Czy model bywa mocno pewny błędnych decyzji? (konkretny przykład)**

_(Twoja odpowiedź tutaj.)_

**4. Na jakie regiony wejścia wskazuje Grad-CAM? (dla Twoich 3 przykładów)**

_(Twoja odpowiedź tutaj.)_

**5. Czy analiza przez perturbacje (occlusion) wspiera tę interpretację?**

_(Twoja odpowiedź tutaj.)_

**6. Czy zachowanie modelu zmienia się po Twojej wybranej modyfikacji wejścia (sekcja 9)?**

_(Twoja odpowiedź tutaj.)_

**7. Co należałoby sprawdzić przed wykorzystaniem tego modelu w praktyce?**

_(Twoja odpowiedź tutaj.)_